# Notebook 05: Discusion, conclusiones y recomendaciones

---

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

PROYECTO = "food_delivery_time_prediction"

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = f"/content/drive/MyDrive/{PROYECTO}"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(f"./{PROYECTO}")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)


def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino)


print("Persistencia en Drive:", EN_DRIVE)
print("Ruta de trabajo:", RUTA)

In [ ]:
X_train = pd.read_csv(os.path.join(RUTA, "X_train.csv"))
X_test = pd.read_csv(os.path.join(RUTA, "X_test.csv"))
y_train = pd.read_csv(os.path.join(RUTA, "y_train.csv")).iloc[:, 0]
y_test = pd.read_csv(os.path.join(RUTA, "y_test.csv")).iloc[:, 0]

NUMERICAS = ["distancia_km", "edad", "calificacion", "pedidos_simultaneos", "estado_vehiculo"]
CATEGORICAS = ["trafico", "clima", "vehiculo", "tipo_pedido", "ciudad", "festivo"]

print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

guardado = joblib.load(os.path.join(RUTA, "modelos_y_metricas.joblib"))
modelo_a = guardado["modelo_a"]
modelo_b = guardado["modelo_b"]
modelo_c = guardado["modelo_c"]
modelo_d = guardado["modelo_d"]
modelo_e = guardado["modelo_e"]
tabla = guardado["tabla_metricas"]

print(tabla[tabla["conjunto"] == "test"].set_index("modelo")[["MSE", "RMSE", "MAE", "R2"]].to_string())

## 6.2 Interpretacion de los indicadores

### Las metricas en el contexto del problema

Las cuatro metricas responden preguntas distintas y conviene leerlas por separado antes de
compararlas entre modelos.

**RMSE, el error tipico en minutos.** Es la metrica que se le puede mostrar a alguien que no
sabe estadistica, porque esta en la misma unidad que la variable: dice en cuantos minutos se
equivoca el modelo, en promedio, dandole mas peso a los errores grandes. La referencia contra
la cual leerlo es la desviacion estandar del tiempo de entrega, que ronda los 9 minutos: ese
es el error que cometeria alguien que siempre predijera el promedio.

**MAE, el error tipico sin castigar los extremos.** Trata todos los errores por igual. Que el
MAE sea menor que el RMSE en los tres modelos indica que hay algunos errores grandes
elevando el RMSE, cosa esperable en cualquier problema con casos atipicos.

**MSE, en minutos al cuadrado.** No es interpretable directamente porque su unidad no
significa nada fisico. Sirve como criterio de optimizacion, que es para lo que existe, y para
comparar modelos entre si.

**R2, la fraccion de la variacion explicada.** Cero significa que el modelo no aporta nada
sobre predecir siempre el promedio. Uno significaria prediccion perfecta.

In [ ]:
comparativa = tabla[tabla["conjunto"] == "test"].set_index("modelo")
desv = y_test.std()
media = y_test.mean()

print(f"Referencia: la desviacion estandar del tiempo de entrega es {desv:.2f} min")
print(f"            el tiempo medio es {media:.2f} min\n")

for nombre, fila in comparativa.iterrows():
    print(f"{nombre}")
    print(f"   RMSE {fila['RMSE']:5.2f} min   ->  {100*fila['RMSE']/media:4.1f} % del tiempo medio, "
          f"y {100*fila['RMSE']/desv:5.1f} % del error de predecir siempre el promedio")
    print(f"   MAE  {fila['MAE']:5.2f} min   ->  la mitad de las entregas se estiman "
          f"con menos de ese margen de error")
    print(f"   R2   {fila['R2']:.4f}      ->  explica el {100*fila['R2']:.1f} % de la variacion\n")

### Comparacion entre los modelos con criterio tecnico

La progresion entre los tres modelos tiene una lectura clara y una consecuencia practica.

**De A a B, el salto grande.** Pasar de usar solo la distancia a usar las once variables
recorta el error tipico en varios minutos y multiplica varias veces la varianza explicada. La
mayor parte de esa mejora viene de las variables categoricas de contexto, no de las numericas
adicionales, como mostro el analisis de aporte individual del notebook 03.

**De B a C, el salto chico.** Agregar cuadrados y productos entre las variables numericas
mejora poco. Eso descarta que las relaciones sean marcadamente curvas dentro del rango
observado, y descarta tambien que falten interacciones entre las numericas.

**De C a D, el segundo salto grande.** Generar los productos por pares sobre la matriz
completa, con las categoricas ya codificadas, recorta el error tipico bastante mas de lo que
habia logrado la expansion polinomica. La diferencia entre ambos esta en el origen de las
interacciones: el modelo C solo genera productos entre variables numericas, mientras que el D los
genera sobre la matriz completa e incluye por lo tanto los productos con las columnas binarias
de las categoricas. La magnitud de la mejora indica que esas combinaciones concentraban una parte
sustantiva de la varianza no explicada.

Traducido al problema: el modelo B suponia que un kilometro cuesta lo mismo con la avenida
libre que dentro de un atasco, y que el clima afecta igual a una ciudad metropolitana que a una
semiurbana. Ninguna de las dos cosas es cierta, y el modelo D es el primero que puede
expresarlo.

**De D a E, la curvatura que el modelo C no habia encontrado.** Reemplazar los terminos
cuadraticos por una base de splines cubicos vuelve a mejorar de forma clara. Eso corrige la
lectura que sugeria el modelo C: las relaciones no eran lineales, sino que un polinomio de grado
dos era demasiado rigido para describirlas. Un spline ajusta un tramo distinto en cada zona del
rango, y esa flexibilidad local es la que faltaba.

**Cual conviene usar.** Depende de para que.

Para **predecir**, el modelo E. Es el que menos se equivoca y su ventaja sobre el resto no es
marginal.

Para **explicar**, el modelo B. Sus coeficientes se leen directamente en minutos y responden la
pregunta del negocio, que es cuanto pesa cada factor. Los modelos D y E tienen mas de mil
coeficientes sobre productos de variables transformadas, y ninguno significa nada por si solo.

No es una contradiccion sino dos usos distintos del mismo experimento. La seccion siguiente
trabaja con el modelo B precisamente porque ahi el proposito es explicar.

### Que dicen los coeficientes

Para leer los coeficientes en unidades reales se reajusta el modelo B sin estandarizar. Asi
cada coeficiente se lee directamente como minutos: cuantos minutos agrega una unidad
adicional de esa variable, o cuantos agrega una categoria respecto de su referencia.

In [ ]:
prep_crudo = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERICAS),
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]), CATEGORICAS),
])

modelo_interpretable = Pipeline([("prep", prep_crudo), ("regresion", LinearRegression())])
modelo_interpretable.fit(X_train, y_train)

nombres_cat = modelo_interpretable.named_steps["prep"].named_transformers_["cat"] \
    .named_steps["codificar"].get_feature_names_out(CATEGORICAS)
nombres = list(NUMERICAS) + list(nombres_cat)

coeficientes = pd.Series(modelo_interpretable.named_steps["regresion"].coef_, index=nombres)

print(f"Ordenada al origen: {modelo_interpretable.named_steps['regresion'].intercept_:.2f} min\n")
print("Variables numericas, minutos por cada unidad adicional:")
print(coeficientes[NUMERICAS].round(3).to_string())
print("\nVariables categoricas, minutos respecto de la categoria de referencia:")
print(coeficientes.drop(NUMERICAS).sort_values(ascending=False).round(2).to_string())

In [ ]:
rango_distancia = X_train["distancia_km"].max() - X_train["distancia_km"].min()
efecto_distancia_total = coeficientes["distancia_km"] * rango_distancia

print("Traduccion a decisiones operativas\n")
print(f"Un kilometro adicional agrega {coeficientes['distancia_km']:.2f} min "
      f"({coeficientes['distancia_km']*60:.0f} segundos)")
print(f"Todo el rango de distancias, {rango_distancia:.1f} km, agrega "
      f"{efecto_distancia_total:.1f} min\n")

equivalencias = {
    "Un pedido simultaneo mas": coeficientes["pedidos_simultaneos"],
    "Un punto mas de calificacion del repartidor": coeficientes["calificacion"],
}
for clave in coeficientes.index:
    if clave.startswith("trafico_") or clave.startswith("festivo_") or clave == "clima_Sunny":
        equivalencias[clave] = coeficientes[clave]

for etiqueta, valor in equivalencias.items():
    km = valor / coeficientes["distancia_km"]
    print(f"{etiqueta:46s} {valor:+6.2f} min  =  {km:+6.1f} km de distancia")

**Lectura de los coeficientes.** La ultima columna traduce cada efecto a la unidad que la
intuicion maneja: cuantos kilometros de distancia equivalen a ese cambio. Es la forma mas
directa de mostrar la magnitud relativa.

Un kilometro adicional agrega apenas una fraccion de minuto, de modo que recorrer todo el
rango de distancias del conjunto suma unos pocos minutos. En cambio salir de un atasco, o
que el dia sea festivo, mueven el tiempo estimado varias veces mas que eso, y equivalen a
decenas de kilometros de distancia.

El coeficiente de la calificacion del repartidor requiere una precision. Es negativo y de
magnitud considerable por unidad, pero la variable se distribuye entre 2,5 y 5 con la mayor parte
de los casos por encima de 4,5. El efecto que produce en la practica es el coeficiente
multiplicado por el rango efectivo de la variable, no por su rango nominal, y por eso su aporte a
la prediccion es menor de lo que sugiere el valor aislado.

## 7.1 Contraste con la hipotesis

La hipotesis planteada en el notebook 01 tenia dos partes, cada una con su criterio de
rechazo. Se evaluan por separado.

In [ ]:
train = X_train.copy()
train["minutos"] = y_train.values

corr_distancia = train["distancia_km"].corr(train["minutos"])
coef_distancia = coeficientes["distancia_km"]

varianza_total = ((train["minutos"] - train["minutos"].mean()) ** 2).sum()
g = train.groupby("trafico", observed=True)["minutos"]
var_trafico = (((g.mean() - train["minutos"].mean()) ** 2) * g.count()).sum() / varianza_total
var_distancia = corr_distancia ** 2

print("PRIMERA PARTE: la distancia predice positivamente el tiempo")
print(f"   correlacion distancia-tiempo   {corr_distancia:+.3f}")
print(f"   coeficiente en el modelo B     {coef_distancia:+.3f} min por km")
print(f"   veredicto: {'CONFIRMADA' if corr_distancia > 0 and coef_distancia > 0 else 'RECHAZADA'}\n")

print("SEGUNDA PARTE: el trafico explica mas varianza que la distancia")
print(f"   varianza explicada por el trafico    {var_trafico:.4f}")
print(f"   varianza explicada por la distancia  {var_distancia:.4f}")
print(f"   razon entre ambas                    {var_trafico/var_distancia:.2f} a 1")
print(f"   veredicto: {'CONFIRMADA' if var_trafico > var_distancia else 'RECHAZADA'}")

### Veredicto

**La primera parte se confirma.** La correlacion entre distancia y tiempo es positiva y el
coeficiente en el modelo multiple tambien lo es. La direccion anticipada era la correcta.

**La segunda parte tambien se confirma, y con un margen amplio.** El trafico explica por si
solo bastante mas varianza que la distancia. La relacion entre ambas cifras cuantifica el
resultado central del experimento.

**El resultado acota el alcance de la primera parte.** Confirmar que la distancia predice
positivamente el tiempo es correcto y a la vez insuficiente para el negocio. El modelo A, que
usa unicamente la distancia, explica una porcion muy pequena de la variacion. Es decir: la
relacion existe, tiene el signo esperado y es estadisticamente clara, pero es demasiado debil
para sostener por si sola un sistema de estimacion. Una hipotesis puede ser verdadera y aun
asi no alcanzar para lo que se necesita.

### Causas de la brecha

Tres razones explican por que la distancia predice tan poco:

**El rango de distancias es estrecho.** El conjunto abarca de uno a treinta kilometros y la
mitad central se concentra entre cinco y catorce. Una variable que apenas varia no puede
explicar mucha variacion en otra.

**La velocidad varia mas que la distancia.** Es el argumento tecnico de la justificacion de
la hipotesis, y los datos lo confirman: entre transito libre y atasco la diferencia de tiempo
supera la que produce recorrer el doble de distancia.

**La distancia es en linea recta, no por calle.** El recorrido real depende del trazado
urbano, de los sentidos de circulacion y de los giros permitidos. Esa diferencia entre
distancia geometrica y distancia efectiva es ruido que el modelo no puede resolver con la
informacion disponible.

## 7.2 Analisis de sobreajuste y subajuste

El diagnostico no se hace mirando una sola cifra sino la trayectoria del error de
entrenamiento y del de prueba a medida que aumenta la complejidad del modelo. Se ajustan
polinomios de grado uno, dos y tres sobre las mismas variables, y se agrega el modelo E como
cuarto punto, que es el mas complejo de todos los ensayados. Lo que interesa es como evolucionan
ambos errores y, sobre todo, la distancia entre ellos.

Un modelo **sobreajustado** muestra un error de entrenamiento que sigue bajando mientras el de
prueba se estanca o empeora: la brecha se abre. Un modelo **subajustado** muestra ambos
errores altos y juntos, y sigue mejorando cuando se le da mas capacidad.

In [ ]:
def preprocesador(numericas, categoricas=None, grado=1):
    pasos_num = [("imputar", SimpleImputer(strategy="median")),
                 ("escalar", StandardScaler())]
    if grado > 1:
        pasos_num.append(("polinomio", PolynomialFeatures(grado, include_bias=False)))
    ramas = [("num", Pipeline(pasos_num), numericas)]
    if categoricas:
        ramas.append(("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ]), categoricas))
    return ColumnTransformer(ramas)


barrido = []
for grado in [1, 2, 3]:
    m = Pipeline([("prep", preprocesador(NUMERICAS, CATEGORICAS, grado)),
                  ("regresion", LinearRegression())])
    m.fit(X_train, y_train)
    r2_tr = r2_score(y_train, m.predict(X_train))
    r2_te = r2_score(y_test, m.predict(X_test))
    barrido.append({
        "modelo": None,
        "grado": grado,
        "variables": m.named_steps["prep"].transform(X_train.head(5)).shape[1],
        "R2_train": r2_tr,
        "R2_test": r2_te,
        "brecha": r2_tr - r2_te,
        "RMSE_test": np.sqrt(mean_squared_error(y_test, m.predict(X_test))),
    })

r2_tr_e = r2_score(y_train, modelo_e.predict(X_train))
r2_te_e = r2_score(y_test, modelo_e.predict(X_test))
barrido.append({
    "modelo": "E. splines",
    "variables": modelo_e.named_steps["interacciones"].transform(
        modelo_e.named_steps["prep"].transform(X_train.head(5))).shape[1],
    "R2_train": r2_tr_e,
    "R2_test": r2_te_e,
    "brecha": r2_tr_e - r2_te_e,
    "RMSE_test": np.sqrt(mean_squared_error(y_test, modelo_e.predict(X_test))),
})

barrido = pd.DataFrame(barrido)
barrido["modelo"] = barrido["modelo"].fillna(
    "polinomio grado " + barrido["grado"].astype("Int64").astype(str))
barrido = barrido[["modelo", "variables", "R2_train", "R2_test", "brecha", "RMSE_test"]]
print(barrido.round(4).to_string(index=False))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4.4))

posicion = range(len(barrido))
etiquetas_x = ["grado 1", "grado 2", "grado 3", "splines"]

ejes[0].plot(posicion, barrido["R2_train"], "o-", color="#101010", label="entrenamiento")
ejes[0].plot(posicion, barrido["R2_test"], "s--", color="#909090", label="prueba")
ejes[0].set_xlabel("Complejidad de la representacion")
ejes[0].set_ylabel("R2")
ejes[0].set_title("Desempeno frente a la complejidad del modelo")
ejes[0].set_xticks(list(posicion))
ejes[0].set_xticklabels(etiquetas_x)
ejes[0].legend()

ejes[1].bar(posicion, barrido["brecha"], color="#707070", width=0.5)
ejes[1].set_xlabel("Complejidad de la representacion")
ejes[1].set_ylabel("R2 train - R2 test")
ejes[1].set_title("Brecha entre entrenamiento y prueba")
ejes[1].set_xticks(list(posicion))
ejes[1].set_xticklabels(etiquetas_x)

plt.tight_layout()
guardar("05_complejidad_y_sobreajuste")
plt.show()

### Diagnostico: subajuste, no sobreajuste

**No hay sobreajuste en ninguno de los cuatro.** La brecha entre entrenamiento y prueba se
mantiene por debajo de una centesima de R2 en todos los casos, e incluso sale negativa en el
modelo mas simple, donde el desempeno sobre prueba resulta marginalmente mejor. Eso no es un
error de calculo: con brechas de ese tamano la diferencia entre conjuntos es ruido de muestreo y
su signo no significa nada. La brecha crece al aumentar la complejidad, que es la direccion
esperada, pero incluso el modelo E, con mas de mil quinientas variables, se mantiene mas de un
orden de magnitud por debajo de lo que indicaria memorizacion.

La razon es el volumen combinado con la regularizacion. Mas de treinta mil observaciones de
entrenamiento dejan poco margen para memorizar, y en los modelos con muchas variables la
penalizacion de Ridge se encarga del resto.

**Hay subajuste, y las dos curvas lo muestran.** Ambas suben juntas al aumentar el grado, sin
separarse. Cuando un modelo mejora en prueba al darle mas capacidad, todavia no agoto lo que
podia extraer de los datos.

**Por que entonces se elige el modelo B y no el de grado tres.** Porque la mejora que aporta
la complejidad adicional es pequena frente a lo que cuesta. Cada grado extra multiplica la
cantidad de variables y vuelve los coeficientes imposibles de interpretar, y el criterio 6.2
de este experimento depende justamente de poder leerlos. En un problema donde la limitacion
es la informacion disponible y no la forma funcional, agregar flexibilidad rinde poco.

### Cuanto de lo que falta es culpa de la familia lineal

El diagnostico de subajuste deja una pregunta abierta: el techo lo impone la informacion
contenida en las variables, o la incapacidad de un modelo lineal para aprovecharla? Son dos
causas distintas y llevan a recomendaciones opuestas.

La forma de separarlas es ajustar un modelo de otra familia sobre exactamente las mismas
variables y la misma particion. Si rinde parecido, la limitacion esta en los datos. Si rinde
bastante mejor, la limitacion esta en la forma funcional. El modelo de arboles con potenciacion
del gradiente no es el objeto de este experimento ni se propone como solucion: se usa
unicamente como instrumento de medicion.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

referencia = Pipeline([
    ("prep", preprocesador(NUMERICAS, CATEGORICAS)),
    ("arboles", HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=400)),
])
referencia.fit(X_train, y_train)

r2_ref_train = r2_score(y_train, referencia.predict(X_train))
r2_ref_test = r2_score(y_test, referencia.predict(X_test))
rmse_ref = np.sqrt(mean_squared_error(y_test, referencia.predict(X_test)))
mejor_lineal = comparativa["R2"].max()

print("Techo alcanzable con las mismas once variables")
print()
print(f"  Mejor modelo lineal (E)         R2 test {mejor_lineal:.4f}")
print(f"  Referencia no lineal (arboles)  R2 test {r2_ref_test:.4f}   "
      f"RMSE {rmse_ref:.2f} min   brecha {r2_ref_train - r2_ref_test:+.4f}")
print()
print(f"  Diferencia atribuible a la familia de modelos: {r2_ref_test - mejor_lineal:+.4f} de R2")

**El resultado ubica el techo.** El modelo de arboles, con las mismas once variables y la misma
particion, explica algo mas de varianza que el mejor de los lineales, pero la diferencia es
mucho menor de lo que era antes de incorporar splines e interacciones. Dicho de otro modo: la
informacion **si estaba** en los datos, y la familia lineal puede aprovechar la mayor parte de
ella siempre que se le construyan las variables adecuadas. Lo que no puede es encontrarlas sola.

Es coherente con lo que mostro el notebook 03. Las variables mas informativas son categoricas y
su efecto depende de la combinacion en la que aparecen: el atasco no pesa igual en una ciudad
semiurbana que en una metropolitana, y el dia festivo altera el comportamiento de todo lo demas.
Un modelo de arboles descubre esas combinaciones y esos umbrales por su cuenta, mientras que uno
lineal necesita que se los construyan como productos y como bases de splines. El experimento
muestra que, hecho ese trabajo, la brecha entre ambas familias se reduce a una fraccion de lo
que era.

Esta comparacion tiene una limitacion. Una ventaja de esa magnitud para los arboles es
consistente con que el conjunto contenga relaciones definidas por umbrales, algo frecuente en
datos de competencia generados de forma parcialmente sintetica. Sobre registros operativos la
diferencia entre ambas familias suele ser menor, de modo que el valor obtenido aca funciona como
cota superior optimista.

**Como se mitigaria el subajuste.** El orden de prioridad cambia a la luz de este resultado:

1. **Construir mejor las variables.** Es lo que hicieron los modelos D y E, y lo que mas rindio
   dentro de la familia lineal sin salir de ella: primero las interacciones, despues los splines.
2. **Considerar un modelo no lineal** si el objetivo pasa a ser predecir en lugar de explicar.
   El costo es perder la lectura directa de los coeficientes, que en este problema tiene valor
   propio.
3. **Incorporar variables ausentes**, como la distancia por calle o el tiempo de preparacion en
   cocina. Sigue siendo util, pero ya no es la explicacion principal del techo.

## 8.1 Conclusiones

**Primera. La distancia predice el tiempo de entrega en la direccion esperada, pero es
insuficiente por si sola.** Un modelo de regresion lineal simple sobre la distancia explica
una porcion muy pequena de la variacion del tiempo, con un error tipico cercano al que se
obtendria prediciendo siempre el promedio. La primera parte de la hipotesis se confirma en
signo y se matiza en magnitud: la relacion existe pero no alcanza para operar.

**Segunda. El contexto de circulacion explica mas que la geometria del recorrido.** El
trafico, medido como variable individual, explica varias veces mas varianza que la distancia,
y el analisis cruzado del notebook 03 lo mostro de forma directa: dentro de un mismo tramo de
distancia, pasar de transito bajo a atasco agrega mas minutos que duplicar la distancia con
el trafico constante. La segunda parte de la hipotesis se confirma.

**Tercera. Lo que limitaba al modelo era como estaban representadas las variables, no la falta
de informacion.** La comparacion entre los cinco modelos lo descompone en pasos: agregar
variables produce el primer salto, elevarlas al cuadrado casi no aporta, agregar los productos
entre categoricas produce el segundo salto, y reemplazar los cuadrados por splines produce el
tercero. Ninguno de los cinco muestra brecha entre entrenamiento y prueba, de modo que el
problema nunca fue el sobreajuste.

Los dos ultimos saltos acotan la lectura del modelo C. Su escaso aporte no permite concluir que
las relaciones sean lineales, sino unicamente que un polinomio global de grado dos no alcanza
para representarlas. Una base de splines, que ajusta un tramo distinto en cada zona del rango,
recupera la curvatura que ese polinomio no podia describir.

**Cuarta. El experimento deja dos modelos utiles, y para cosas distintas.** El modelo E, con
splines e interacciones, es el que menos se equivoca y el que conviene si el objetivo es
predecir. El modelo B, mucho mas simple, es el unico cuyos coeficientes se leen directamente en
minutos, y es el que sirve para explicarle a la operacion cuanto pesa cada factor. Elegir uno
solo obligaria a renunciar a una de las dos cosas, y nada en el problema exige esa renuncia.

**Quinta, sobre lo que el experimento significa para el negocio.** El resultado tiene una
consecuencia directa sobre como deberia construirse un estimador de tiempo de entrega:
alimentarlo con la distancia es barato pero rinde poco, y alimentarlo con trafico y clima en
tiempo real es lo que produce la diferencia. Para una operacion en Santa Cruz de la Sierra,
donde la congestion en los anillos y la temporada de lluvias cambian las condiciones de
circulacion de forma marcada, esa distincion define si la promesa al cliente se cumple.

## 8.2 Recomendaciones

**Primera. Incorporar el tiempo de preparacion en cocina.** El conjunto registra cuando se
toma el pedido y cuando se recoge, pero ese intervalo no se uso como predictor porque en el
momento de prometer un tiempo al cliente todavia no se conoce. La recomendacion concreta es
modelarlo aparte, a partir del restaurante y del tipo de pedido, y sumar esa estimacion a la
del trayecto. Es la variable ausente que mejor explica el subajuste diagnosticado en el 7.2:
hoy toda la variacion de la cocina cae dentro del error del modelo.

**Segunda. Reemplazar la distancia en linea recta por distancia de ruta.** La distancia
calculada con Haversine ignora el trazado de calles, los sentidos de circulacion y los giros
prohibidos. Consultar un servicio de rutas devolveria la distancia efectiva y, en el mismo
llamado, una estimacion de duracion que ya incorpora la geometria de la red vial. Ataca la
tercera causa de la brecha identificada en el 7.1.

**Tercera. Registrar el trafico con granularidad continua en lugar de cuatro niveles.** La
variable mas informativa del experimento esta discretizada en cuatro categorias, lo que
descarta toda la variacion dentro de cada nivel. Una medida continua, como la velocidad media
del tramo en el momento del pedido, conservaria esa informacion. Dado que el trafico es el
predictor mas fuerte, mejorar su medicion es donde mas rinde el esfuerzo.

**Cuarta. Recalibrar con datos locales antes de cualquier uso operativo.** Los registros
provienen de ciudades de India y las magnitudes concretas no se trasladan a Santa Cruz: las
velocidades medias, las distancias tipicas y el efecto de la temporada de lluvias son
distintos. La estructura del problema si se traslada, y las conclusiones sobre que variables
importan tambien, pero los coeficientes deben reestimarse sobre entregas locales.

**Quinta. Depurar las interacciones en lugar de generarlas todas.** Los modelos D y E construyen
todos los productos por pares sin criterio, y esa exhaustividad es la que obliga a regularizar
con Ridge. Un paso siguiente razonable es identificar cuales concentran la mejora, usando la
magnitud de los coeficientes como guia, y armar un modelo reducido que conserve solo esas. Se
recuperaria buena parte de la interpretabilidad perdida sin volver al desempeno del modelo B.

Tambien conviene ajustar la cantidad de nudos de los splines por validacion cruzada en lugar de
fijarla, que es lo que hace este experimento.

**Sexta. Evaluar un modelo no lineal si la prioridad pasa a ser predecir.** La comparacion de la
seccion 7.2 muestra que queda margen considerable por encima de lo que alcanza la familia
lineal. La decision no es tecnica sino de producto: un modelo de arboles estima mejor, pero no
permite responderle a un operador por que el sistema prometio veintiocho minutos y no veinte.

## Cierre

El experimento respondio la pregunta que planteo: la distancia importa, pero el trafico
importa mas, y esa diferencia tiene consecuencias directas sobre como debe construirse un
sistema de estimacion de tiempos de entrega.

Las figuras generadas quedaron en la carpeta `splits/` del proyecto y los modelos entrenados
en `modelos_y_metricas.joblib`, listos para el informe escrito.